# Lab 6: Road Sign Segmentation (VS Code, no YOLO, no MMDetection)

This notebook uses `torchvision` + `DeepLabV3` for multiclass segmentation of 8 road sign classes.
It is designed for local run in VS Code.

## 0) Local setup (run once in VS Code terminal)

```bash
python -m pip install -U pip
python -m pip install torch torchvision torchaudio
python -m pip install opencv-python pillow pandas matplotlib tqdm scikit-learn
```

After install: restart notebook kernel.

In [5]:
from __future__ import annotations

import json
import random
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "sign_dataset"
TEST_IMAGES_DIR = PROJECT_ROOT / "test_images_lab6"
REPORT_DIR = PROJECT_ROOT / "reports" / "lab6_torchvision"
VIS_DIR = REPORT_DIR / "visualizations"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

IMG_TRAIN_DIR = DATA_ROOT / "images" / "train"
IMG_VAL_DIR = DATA_ROOT / "images" / "val"
LBL_TRAIN_DIR = DATA_ROOT / "labels" / "train"
LBL_VAL_DIR = DATA_ROOT / "labels" / "val"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python root:", PROJECT_ROOT)
print("Device:", device)
print("Dataset exists:", DATA_ROOT.exists())
print("Test images exists:", TEST_IMAGES_DIR.exists())

Python root: c:\Users\wiad_\PycharmProjects\ItmoCv
Device: cuda
Dataset exists: True
Test images exists: True


In [6]:
print(device)
print(torch.cuda.is_available())


cuda
True


In [7]:
def parse_yolo_poly_line(line: str):
    parts = line.strip().split()
    if len(parts) < 7:
        return None
    cls_id = int(float(parts[0]))
    coords = [float(x) for x in parts[1:]]
    if len(coords) % 2 != 0:
        return None
    pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
    return cls_id, pts


def class_frequency(label_dir: Path):
    freq = Counter()
    for txt in sorted(label_dir.glob("*.txt")):
        with txt.open("r", encoding="utf-8") as f:
            for line in f:
                parsed = parse_yolo_poly_line(line)
                if parsed is None:
                    continue
                cls, _ = parsed
                freq[cls] += 1
    return freq


mapping_path = PROJECT_ROOT / "reports" / "lab6" / "class_mapping.json"
if mapping_path.exists():
    mapping_data = json.loads(mapping_path.read_text(encoding="utf-8"))
    selected_raw_classes = [int(x) for x in mapping_data["selected_raw_classes"]]
else:
    freq = class_frequency(LBL_TRAIN_DIR)
    selected_raw_classes = [c for c, _ in freq.most_common(8)]

raw_to_model = {raw: i + 1 for i, raw in enumerate(selected_raw_classes)}  # 0 is background
id_to_name = {0: "background"}
for raw, idx in raw_to_model.items():
    id_to_name[idx] = f"class_{raw}"

num_classes = 1 + len(selected_raw_classes)
print("Selected raw classes:", selected_raw_classes)
print("Model classes:", id_to_name)
print("num_classes:", num_classes)

Selected raw classes: [3, 8, 1, 10, 6, 2, 4, 13]
Model classes: {0: 'background', 1: 'class_3', 2: 'class_8', 3: 'class_1', 4: 'class_10', 5: 'class_6', 6: 'class_2', 7: 'class_4', 8: 'class_13'}
num_classes: 9


In [8]:



class RoadSignsSegDataset(Dataset):
    def __init__(self, images_dir: Path, labels_dir: Path, raw_to_model: dict[int, int], image_size: int = 512):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.raw_to_model = raw_to_model
        self.image_paths = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
        self.image_size = image_size
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.image_paths)

    def _build_mask(self, image_w: int, image_h: int, label_path: Path):
        mask = np.zeros((image_h, image_w), dtype=np.uint8)
        if not label_path.exists():
            return mask

        with label_path.open("r", encoding="utf-8") as f:
            for line in f:
                parsed = parse_yolo_poly_line(line)
                if parsed is None:
                    continue
                raw_cls, pts_norm = parsed
                if raw_cls not in self.raw_to_model:
                    continue
                pts = pts_norm.copy()
                pts[:, 0] = np.clip(pts[:, 0] * image_w, 0, image_w - 1)
                pts[:, 1] = np.clip(pts[:, 1] * image_h, 0, image_h - 1)
                pts = pts.astype(np.int32)
                if len(pts) >= 3:
                    cv2.fillPoly(mask, [pts], color=int(self.raw_to_model[raw_cls]))
        return mask

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        lbl_path = self.labels_dir / f"{img_path.stem}.txt"

        with Image.open(img_path) as im:
            img = im.convert("RGB")

        w, h = img.size
        mask = self._build_mask(w, h, lbl_path)

        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = cv2.resize(mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)

        img_t = self.to_tensor(img)
        mask_t = torch.from_numpy(mask).long()
        return img_t, mask_t, img_path.name


train_ds = RoadSignsSegDataset(IMG_TRAIN_DIR, LBL_TRAIN_DIR, raw_to_model, image_size=512)
val_ds = RoadSignsSegDataset(IMG_VAL_DIR, LBL_VAL_DIR, raw_to_model, image_size=512)

train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True,
    num_workers=0,                      # <- important in notebook on Windows
    pin_memory=(device.type == "cuda"),
    persistent_workers=False
)
val_loader = DataLoader(
    val_ds, batch_size=4, shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    persistent_workers=False
)

print("Train images:", len(train_ds))
print("Val images:", len(val_ds))

Train images: 2054
Val images: 127


In [9]:
model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)
model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
if model.aux_classifier is not None:
    model.aux_classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 8
print("Model ready. EPOCHS:", EPOCHS)

Model ready. EPOCHS: 8


In [10]:
device = torch.device("cuda:0")
torch.cuda.set_per_process_memory_fraction(7.5 / 8.0, device=0)


use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def train_one_epoch(model, loader, optimizer, criterion, device, scaler, accum_steps=1, log_every=50):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (imgs, masks, _) in enumerate(tqdm(loader, leave=False), start=1):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            out = model(imgs)["out"]
            loss = criterion(out, masks) / accum_steps

        scaler.scale(loss).backward()

        if step % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * accum_steps * imgs.size(0)

        if use_amp and (step % log_every == 0):
            alloc = torch.cuda.memory_allocated() / 1024**3
            peak = torch.cuda.max_memory_allocated() / 1024**3
            print(f"[train step {step}] alloc={alloc:.2f}GB peak={peak:.2f}GB")

    # flush last partial accumulation
    if len(loader) % accum_steps != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    return total_loss / len(loader.dataset)


history = []
best_val_loss = float("inf")
best_path = REPORT_DIR / "best_deeplabv3_sign8.pth"

for epoch in range(1, EPOCHS + 1):
    if use_amp:
        torch.cuda.reset_peak_memory_stats()

    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion, device, scaler,
        accum_steps=2,   # increase to reduce VRAM usage (effective batch = batch_size * accum_steps)
        log_every=50
    )

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks, _ in tqdm(val_loader, leave=False):
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(imgs)["out"]
                loss = criterion(out, masks)

            val_loss += loss.item() * imgs.size(0)

    val_loss /= len(val_loader.dataset)

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    if use_amp:
        peak = torch.cuda.max_memory_allocated() / 1024**3
        print(f"Epoch {epoch:02d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, peak_vram={peak:.2f}GB")
        torch.cuda.empty_cache()
    else:
        print(f"Epoch {epoch:02d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_path)

pd.DataFrame(history).to_csv(REPORT_DIR / "train_history.csv", index=False)
print("Best model:", best_path)


C:\Users\wiad_\AppData\Local\Temp\ipykernel_12572\2949973621.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


  0%|          | 0/514 [00:00<?, ?it/s]

C:\Users\wiad_\AppData\Local\Temp\ipykernel_12572\2949973621.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[train step 50] alloc=0.52GB peak=2.90GB
[train step 100] alloc=0.52GB peak=2.90GB
[train step 150] alloc=0.52GB peak=2.90GB
[train step 200] alloc=0.52GB peak=2.90GB
[train step 250] alloc=0.52GB peak=2.90GB
[train step 300] alloc=0.52GB peak=2.90GB
[train step 350] alloc=0.52GB peak=2.90GB
[train step 400] alloc=0.52GB peak=2.90GB
[train step 450] alloc=0.52GB peak=2.90GB
[train step 500] alloc=0.52GB peak=2.90GB


  0%|          | 0/32 [00:00<?, ?it/s]

...

Epoch 07: train_loss=0.0092, val_loss=0.0672, peak_vram=2.94GB


  0%|          | 0/514 [00:00<?, ?it/s]

[train step 50] alloc=0.56GB peak=2.94GB
[train step 100] alloc=0.56GB peak=2.94GB
[train step 150] alloc=0.56GB peak=2.94GB
[train step 200] alloc=0.56GB peak=2.94GB
[train step 250] alloc=0.56GB peak=2.94GB
[train step 300] alloc=0.56GB peak=2.94GB
[train step 350] alloc=0.56GB peak=2.94GB
[train step 400] alloc=0.56GB peak=2.94GB
[train step 450] alloc=0.56GB peak=2.94GB
[train step 500] alloc=0.56GB peak=2.94GB


  0%|          | 0/32 [00:00<?, ?it/s]

Epoch 08: train_loss=0.0080, val_loss=1.4754, peak_vram=2.94GB
Best model: c:\Users\wiad_\PycharmProjects\ItmoCv\reports\lab6_torchvision\best_deeplabv3_sign8.pth


In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

def binary_metrics(pred_mask: np.ndarray, gt_mask: np.ndarray):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, np.logical_not(gt)).sum()
    fn = np.logical_and(np.logical_not(pred), gt).sum()
    union = np.logical_or(pred, gt).sum()

    iou = float(tp / union) if union > 0 else 1.0
    precision = float(tp / (tp + fp)) if (tp + fp) > 0 else 1.0
    recall = float(tp / (tp + fn)) if (tp + fn) > 0 else 1.0
    l2 = float(np.sqrt(np.mean((pred.astype(np.float32) - gt.astype(np.float32)) ** 2)))
    return iou, precision, recall, l2


rows = []
with torch.no_grad():
    for imgs, masks, names in tqdm(val_loader):
        logits = model(imgs.to(device))["out"]
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        gts = masks.numpy()

        for i in range(len(names)):
            pred_bin = (preds[i] > 0).astype(np.uint8)
            gt_bin = (gts[i] > 0).astype(np.uint8)
            iou, precision, recall, l2 = binary_metrics(pred_bin, gt_bin)
            rows.append({
                "image": names[i],
                "iou": iou,
                "precision": precision,
                "recall": recall,
                "l2": l2,
            })

val_metrics_df = pd.DataFrame(rows)
val_summary = {
    "mean_iou": float(val_metrics_df["iou"].mean()),
    "mean_precision": float(val_metrics_df["precision"].mean()),
    "mean_recall": float(val_metrics_df["recall"].mean()),
    "mean_l2": float(val_metrics_df["l2"].mean()),
    "pct_iou_ge_05": float((val_metrics_df["iou"] >= 0.5).mean() * 100.0),
    "pct_iou_ge_075": float((val_metrics_df["iou"] >= 0.75).mean() * 100.0),
    "pct_iou_ge_09": float((val_metrics_df["iou"] >= 0.9).mean() * 100.0),
}

val_metrics_df.to_csv(REPORT_DIR / "val_per_image_metrics.csv", index=False)
pd.DataFrame([val_summary]).to_csv(REPORT_DIR / "val_summary_metrics.csv", index=False)
val_summary

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\wiad_\\PycharmProjects\\ItmoCv\\reports\\lab6_torchvision\\best_deeplabv3_sign8.pth'

In [ ]:
def overlay_prediction(image_rgb: np.ndarray, pred_mask: np.ndarray):
    color = np.zeros_like(image_rgb)
    color[pred_mask > 0] = np.array([255, 80, 80], dtype=np.uint8)
    out = cv2.addWeighted(image_rgb, 0.75, color, 0.25, 0)
    return out


test_image_paths = sorted([p for p in TEST_IMAGES_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])[:10]
print("Selected test images:", len(test_image_paths))
for p in test_image_paths:
    print(" -", p.name)

model.eval()
test_rows = []
for p in test_image_paths:
    img = Image.open(p).convert("RGB")
    orig = np.array(img)
    img_resized = img.resize((512, 512), Image.BILINEAR)
    inp = transforms.ToTensor()(img_resized).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(inp)["out"]
        pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

    pred_orig_size = cv2.resize(pred, (orig.shape[1], orig.shape[0]), interpolation=cv2.INTER_NEAREST)
    vis = overlay_prediction(orig, pred_orig_size)
    out_path = VIS_DIR / f"pred_{p.stem}.png"
    Image.fromarray(vis).save(out_path)
    test_rows.append({"image": p.name, "visualization": str(out_path)})

pd.DataFrame(test_rows).to_csv(REPORT_DIR / "test10_visualizations.csv", index=False)
print("Saved visualizations to:", VIS_DIR)

In [ ]:
# Optional quantitative metrics for personal 10 photos.
# Expected label format: test_images_lab6/labels/<image_stem>.txt
# Same YOLO polygon format as training labels.

TEST_LABELS_DIR = TEST_IMAGES_DIR / "labels"
if not TEST_LABELS_DIR.exists():
    print("No GT labels for personal photos:", TEST_LABELS_DIR)
    print("Only visual quality check is available for personal photos.")
else:
    metric_rows = []
    model.eval()
    for p in test_image_paths:
        img = Image.open(p).convert("RGB")
        w, h = img.size

        ds_tmp = RoadSignsSegDataset(TEST_IMAGES_DIR, TEST_LABELS_DIR, raw_to_model, image_size=512)
        gt_mask = ds_tmp._build_mask(w, h, TEST_LABELS_DIR / f"{p.stem}.txt")

        inp = transforms.ToTensor()(img.resize((512, 512), Image.BILINEAR)).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = torch.argmax(model(inp)["out"], dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        pred = cv2.resize(pred, (w, h), interpolation=cv2.INTER_NEAREST)

        pred_bin = (pred > 0).astype(np.uint8)
        gt_bin = (gt_mask > 0).astype(np.uint8)
        iou, precision, recall, l2 = binary_metrics(pred_bin, gt_bin)
        metric_rows.append({"image": p.name, "iou": iou, "precision": precision, "recall": recall, "l2": l2})

    test_metrics_df = pd.DataFrame(metric_rows)
    test_summary = {
        "mean_iou": float(test_metrics_df["iou"].mean()),
        "mean_precision": float(test_metrics_df["precision"].mean()),
        "mean_recall": float(test_metrics_df["recall"].mean()),
        "mean_l2": float(test_metrics_df["l2"].mean()),
        "pct_iou_ge_05": float((test_metrics_df["iou"] >= 0.5).mean() * 100.0),
        "pct_iou_ge_075": float((test_metrics_df["iou"] >= 0.75).mean() * 100.0),
        "pct_iou_ge_09": float((test_metrics_df["iou"] >= 0.9).mean() * 100.0),
    }
    test_metrics_df.to_csv(REPORT_DIR / "test10_per_image_metrics.csv", index=False)
    pd.DataFrame([test_summary]).to_csv(REPORT_DIR / "test10_summary_metrics.csv", index=False)
    test_summary

## Output files

Main outputs are saved in:
- `reports/lab6_torchvision/train_history.csv`
- `reports/lab6_torchvision/val_per_image_metrics.csv`
- `reports/lab6_torchvision/val_summary_metrics.csv`
- `reports/lab6_torchvision/test10_visualizations.csv`
- `reports/lab6_torchvision/test10_per_image_metrics.csv` (if GT exists)
- `reports/lab6_torchvision/test10_summary_metrics.csv` (if GT exists)
